In [1]:
#Importing Libraries

from collections import Counter
import json
from pathlib import Path
import random
import pandas as pd
import spacy

In [2]:
# Confirming and setting project path

BASE_DIR = Path.cwd().parent
RAW_DIR = BASE_DIR / "Data" / "Raw" #Contains datasets csv
PROJECT_ROOT = BASE_DIR / "Data" / "Processed"
SPLIT_DATA = BASE_DIR / "Data" / "Splits"
SRC = BASE_DIR / "Src" / "Training"
MODEL_PATH = BASE_DIR / "Models" / "spacy_baseline"
print(BASE_DIR)
print(RAW_DIR)
print(PROJECT_ROOT)
print(SPLIT_DATA)
print(SRC)
print(MODEL_PATH)

D:\PycharmProjects\DS Projects\Healthcare NLP Auto-Labeling System_Final
D:\PycharmProjects\DS Projects\Healthcare NLP Auto-Labeling System_Final\Data\Raw
D:\PycharmProjects\DS Projects\Healthcare NLP Auto-Labeling System_Final\Data\Processed
D:\PycharmProjects\DS Projects\Healthcare NLP Auto-Labeling System_Final\Data\Splits
D:\PycharmProjects\DS Projects\Healthcare NLP Auto-Labeling System_Final\Src\Training
D:\PycharmProjects\DS Projects\Healthcare NLP Auto-Labeling System_Final\Models\spacy_baseline


In [6]:
# Load json file

with open(RAW_DIR/'synthetic_notes.json', "r", encoding="utf-8") as file:
    data = json.load(file)

record = data[0]

print(record)
print('\n')
print("ID:")
print(record["id"])

print("\nTEXT:")
print(record["text"])

print("\nENTITIES:")
for entity in record["entities"]:
    print(entity)

{'id': 1, 'text': 'Known case of migraine. Advised to take Oxycodone 20 mg daily.', 'entities': [{'start': 14, 'end': 22, 'label': 'DISEASE'}, {'start': 40, 'end': 49, 'label': 'MEDICATION'}, {'start': 50, 'end': 61, 'label': 'DOSAGE'}]}


ID:
1

TEXT:
Known case of migraine. Advised to take Oxycodone 20 mg daily.

ENTITIES:
{'start': 14, 'end': 22, 'label': 'DISEASE'}
{'start': 40, 'end': 49, 'label': 'MEDICATION'}
{'start': 50, 'end': 61, 'label': 'DOSAGE'}


In [15]:
# Test Tags and Tokens

from nltk.tokenize import word_tokenize

text = record["text"]

tokens = word_tokenize(text)

for token in tokens:

    start = text.find(token, current_pos)
    end = start + len(token)

    tag = "O"

    for entity in record["entities"]:

        if start >= entity["start"] and end <= entity["end"]:

            if start == entity["start"]:
                tag = f"B-{entity['label']}"
            else:
                tag = f"I-{entity['label']}"

            break

    print(f"{token:<20} {tag}")

    current_pos = end

Known                O
case                 O
of                   O
migraine             B-DISEASE
.                    O
Advised              O
to                   O
take                 O
Oxycodone            B-MEDICATION
20                   B-DOSAGE
mg                   I-DOSAGE
daily                I-DOSAGE
.                    O


In [14]:
#Helper Function

from nltk.tokenize import word_tokenize

def convert_record_to_bio(record):
    text = record["text"]
    tokens = word_tokenize(text)

    bio_rows = []
    current_pos = 0

    for token in tokens:
        # Find token position starting from current_pos
        start = text.find(token, current_pos)
        end = start + len(token)

        tag = "O"

        # Check if token falls inside any entity span
        for entity in record["entities"]:
            if start >= entity["start"] and end <= entity["end"]:
                if start == entity["start"]:
                    tag = f"B-{entity['label']}"
                else:
                    tag = f"I-{entity['label']}"
                break

        bio_rows.append((token, tag))
        current_pos = end

    return bio_rows


In [10]:
bio_rows = convert_record_to_bio(data[0])

for row in bio_rows:
    print(row)

('Known', 'O')
('case', 'O')
('of', 'O')
('migraine', 'B-DISEASE')
('.', 'O')
('Advised', 'O')
('to', 'O')
('take', 'O')
('Oxycodone', 'B-MEDICATION')
('20', 'B-DOSAGE')
('mg', 'I-DOSAGE')
('daily', 'I-DOSAGE')
('.', 'O')


In [16]:
for i in range(1):
    print(f"\n===== Record {i} =====")

    bio_rows = convert_record_to_bio(data[i])

    for row in bio_rows:
        print(row)


===== Record 0 =====
('Known', 'O')
('case', 'O')
('of', 'O')
('migraine', 'B-DISEASE')
('.', 'O')
('Advised', 'O')
('to', 'O')
('take', 'O')
('Oxycodone', 'B-MEDICATION')
('20', 'B-DOSAGE')
('mg', 'I-DOSAGE')
('daily', 'I-DOSAGE')
('.', 'O')
